# Table 3 Panel A: Firm-Specific Regressions

Each firm is estimated separately with `asreg`; the paper-reported t-statistics are one-sample tests over the firm-level coefficient distribution. A companion table reports coefficient quartiles and adjusted R-squared distribution.

In [ ]:
version 18
set more off
adopath ++ "../../../../reproductions/resources/ado"

use "data/reg.dta", clear
est clear

* Check the realized endpoints and the stricter interpretation that lead/lag
* data must also lie inside the pooled 1987-2004 outcome window.
count if fyear == 1987
display "Eligible 1987 outcome rows = " r(N)
count if fyear == 2004
display "Eligible 2004 outcome rows = " r(N)
preserve
keep if inrange(fyear, 1988, 2003)
bysort gvkey: egen strict_n = count(fyear)
bysort gvkey: keep if _n == 1
count if strict_n >= 8
quietly summarize strict_n if strict_n >= 8
display "Strict-window firm-years = " %9.0f r(sum) "; firms = " %9.0f r(N)
restore

* asreg estimates separately by gvkey, so there is no single pooled e(sample).
* Every handoff row has complete model fields; precompute counts on these exact
* input rows, then retain only groups with a complete native fit result below.
quietly count if missing(d_wc_scaled, cfo_lag, cfo_cur, cfo_lead)
if r(N) > 0 {
    display as error "Unexpected missing model inputs in the SAS handoff."
    exit 459
}
gen byte __count_original = inrange(fyear, 1987, 1999)
gen byte __count_added = inrange(fyear, 2000, 2004)
bysort gvkey: egen long __fit_original_n = total(__count_original)
bysort gvkey: egen long __fit_added_n = total(__count_added)
gen int __added_year = fyear if __count_added
bysort gvkey: egen int __fit_last_added = max(__added_year)

* Estimate one OLS regression per firm and retain package-generated fit statistics.
asreg d_wc_scaled cfo_lag cfo_cur cfo_lead, by(gvkey) minimum(8)
keep if !missing(_b_cons, _b_cfo_lag, _b_cfo_cur, _b_cfo_lead, _adjR2)
bysort gvkey: keep if _n == 1
quietly count
local s1_successful_firms = r(N)
quietly summarize __fit_original_n, meanonly
local s1_original_n = r(sum)
quietly summarize __fit_added_n, meanonly
local s1_added_n = r(sum)
quietly summarize __fit_last_added if __fit_added_n > 0, meanonly
local s1_last_added = r(max)
display "EXT_SAMPLE s1 " %12.0f `s1_original_n' " " %12.0f `s1_added_n' " " %04.0f `s1_last_added'

display "Successful firm-level fits = " %9.0f `s1_successful_firms'
label variable _b_cons "Intercept"
label variable _b_cfo_lag "CFO(t-1)"
label variable _b_cfo_cur "CFO(t)"
label variable _b_cfo_lead "CFO(t+1)"
label variable _adjR2 "Adjusted R-squared"

* Built-in mean uses one estimate per firm; t statistics remain in the ordinary log, while the export uses template p-values.
mean _b_cons _b_cfo_lag _b_cfo_cur _b_cfo_lead
est store firmmeans
esttab firmmeans using "tables/table3_panel_a_crossfirm_means.md", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) replace label noobs
esttab firmmeans using "tables/table3_panel_a_crossfirm_means.rtf", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) replace label noobs
esttab firmmeans using "tables/table3_panel_a_crossfirm_means.csv", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) replace label noobs
esttab firmmeans using "tables/table3_panel_a_crossfirm_means.tex", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) replace label noobs

* The source reports t-statistics; retain native one-sample t-tests in the log.
ttest _b_cons == 0
ttest _b_cfo_lag == 0
ttest _b_cfo_cur == 0
ttest _b_cfo_lead == 0
est clear

* Companion table reports the full coefficient and adjusted-R-squared distributions.
estpost tabstat _b_cons _b_cfo_lag _b_cfo_cur _b_cfo_lead _adjR2, statistics(N mean sd p25 p50 p75) columns(statistics)
esttab using "tables/table3_panel_a_distribution.md", cells("count(fmt(0)) mean(fmt(3)) sd(fmt(3)) p25(fmt(3)) p50(fmt(3)) p75(fmt(3))") replace label noobs
esttab using "tables/table3_panel_a_distribution.rtf", cells("count(fmt(0)) mean(fmt(3)) sd(fmt(3)) p25(fmt(3)) p50(fmt(3)) p75(fmt(3))") replace label noobs
esttab using "tables/table3_panel_a_distribution.csv", cells("count(fmt(0)) mean(fmt(3)) sd(fmt(3)) p25(fmt(3)) p50(fmt(3)) p75(fmt(3))") replace label noobs
esttab using "tables/table3_panel_a_distribution.tex", cells("count(fmt(0)) mean(fmt(3)) sd(fmt(3)) p25(fmt(3)) p50(fmt(3)) p75(fmt(3))") replace label noobs
est clear
